# Step 4: Retrieval

Docs: https://platform.openai.com/docs/guides/function-calling

In [2]:
import json
import os

import requests
from openai import OpenAI
from pydantic import BaseModel, Field

In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
def search_kb(question: str):
    # Load the knowledge base
    with open("kb.json", 'r')as f:
        return json.load(f)

In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_kb",
            "description": "Get the answer to user's question from the knowledge base provided",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": { "type": "string" },
                },
                "required": ["question"],
                "additionalProperties": False,
            },
            "strict": True,
        }
    }
]

system_prompt = "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."

In [9]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the return policy?"},
]

In [10]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
)

In [11]:
completion.model_dump()

{'id': 'chatcmpl-DdYYwdpOWj2owg8CCCqtQ5YblNABh',
 'choices': [{'finish_reason': 'tool_calls',
   'index': 0,
   'logprobs': None,
   'message': {'content': None,
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_qLOW9cHEidPyd1q0SKVvll01',
      'function': {'arguments': '{"question":"What is the return policy?"}',
       'name': 'search_kb'},
      'type': 'function'}]}}],
 'created': 1778320022,
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': 'fp_54f26dc974',
 'usage': {'completion_tokens': 20,
  'prompt_tokens': 74,
  'total_tokens': 94,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}}

In [26]:
def call_function(name, args):
    if name == "search_kb":
        return search_kb(**args)


for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    messages.append(completion.choices[0].message)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [27]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant that answers questions from the knowledge base about our e-commerce store.'},
 {'role': 'user',
  'content': 'I bought some items but now i want to return it. Is there any way to return it?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_qLOW9cHEidPyd1q0SKVvll01', function=Function(arguments='{"question":"What is the return policy?"}', name='search_kb'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_qLOW9cHEidPyd1q0SKVvll01',
  'content': '{"records": [{"id": 1, "question": "What is the return policy?", "answer": "Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days."}, {"id": 2, "question": "Do you ship internationally?", "answer": "Yes, we ship to over 50 countries worldwide. Int

In [16]:
class KBResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    source: int = Field(description="The record id of the answer.")

In [17]:
completion2 = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    response_format=KBResponse,
)

In [19]:
final_response = completion2.choices[0].message.parsed
print(final_response.answer)
print(final_response.source)

Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.
1


In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the weather in Tokyo?"},
]

completion_3 = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
)

completion_3.choices[0].message.content

"I can't provide real-time weather information. I recommend checking a reliable weather website or app for the most up-to-date weather in Tokyo."

In [28]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I bought some items but now i want to return it. Is there any way to return it?"},
]

completion_4 = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    response_format=KBResponse,
)

completion_4.choices[0].message.content

In [29]:
print(completion_4.choices[0].message.content)

None


In [25]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant that answers questions from the knowledge base about our e-commerce store.'},
 {'role': 'user',
  'content': 'I bought some items but now i want to return it. Is there any way to return it?'}]